<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Transformación Numérica usando PCA (Análisis de Componentes Principales)</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

El **Análisis de Componentes Principales (PCA)** es una técnica de reducción de dimensionalidad que transforma un conjunto de variables posiblemente correlacionadas en un conjunto de variables no correlacionadas llamadas *componentes principales*. PCA es ampliamente utilizado para visualizar datos, eliminar redundancia y mejorar el desempeño de modelos de machine learning.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA


Recordemos que dada cualquier variable o característica, la cantidad de información en ella es proporcional a su varianza.

- si todos los datos son constantes, entonces su varianza es cero y también lo es su contenido de información. Si tenemos datos de muy alta dimensión, podemos reducir su dimensionalidad proyectándolo a lo largo de direcciones (o vectores) tales que la varianza a lo largo de la dirección elegida sea máxima para preservar la mayor cantidad de información posible en los datos.

El **Análisis de Componentes Principales** es un método para encontrar direcciones ortogonales de máxima varianza en los datos, de modo que si los datos se proyectan en esas direcciones, la varianza de los datos proyectados es máxima. La dimensionalidad de los datos puede reducirse proyectándolos en esas direcciones. Esta proyección a lo largo de la dirección de máxima varianza da la mínima pérdida de información.

Encontrar la dirección de máxima varianza para un conjunto de datos dado corresponde a encontrar el vector propio de la matriz de covarianza de los datos, esto lleva a una mínima pérdida de información si los datos se proyectan en esa dirección.


**Deducción**

Supongamos que se nos dan $N$ puntos de datos de $d$ dimensiones $\mathbf x_i, i=1...N$. Queremos encontrar el vector de dirección $\mathbf w$ tal que la proyección $z_i=\mathbf w^T \mathbf x_i$ para un punto $\mathbf x_i$ tenga máxima varianza. Para encontrar la dirección de máxima varianza, necesitaremos desarrollar una fórmula matemática que nos ayude a calcular la varianza de los datos después de haber sido proyectados a lo largo de cierta dirección para que podamos buscar la dirección de máxima varianza optimizando esa fórmula.

Calculemos la varianza después de proyectar los datos a lo largo de $\mathbf w$. Sabemos que la varianza de los valores $z_i, i=1...N$ es el valor esperado (promedio) de la desviación cuadrada $(z_i-\mu_z)$ respecto al valor medio $\mu_z = \frac{1}{N}\sum_{i=1}^{N}z_i=\frac{1}{N}\sum_{i=1}^{N}\mathbf w^T\mathbf x_i=\mathbf w^T\frac{1}{N}\sum_{i=1}^{N}\mathbf x_i=\mathbf w^T\mu_x$ donde $\mathbf{\mu_x}=\frac{1}{N}\sum_{i=1}^{N}\mathbf x_i$ es el vector de $d$ dimensiones de valores promedio de todos los puntos de datos a lo largo de cada una de las $d$ dimensiones. Por lo tanto,

$$var(z)=var(\mathbf w^T \mathbf x)=\frac{1}{N}\sum_{i=1}^{N}[(\mathbf w^T \mathbf x_i-\mathbf w^T\mathbf\mu_x)^2]$$

$$=\frac{1}{N}\sum_{i=1}^{N}[(\mathbf w^T\mathbf x_i-\mathbf w^T\mathbf\mu_x)(\mathbf w^T\mathbf x_i-\mathbf w^T\mathbf\mu_x)]$$

$$=\frac{1}{N}\sum_{i=1}^{N}[(\mathbf w^T\mathbf x_i-\mathbf w^T\mathbf\mu_x)(\mathbf x_i^T\mathbf w-\mathbf \mu_x^T \mathbf w)]$$

$$=\frac{1}{N}\sum_{i=1}^{N}[\mathbf w^T(\mathbf x_i-\mathbf \mu_x)(\mathbf x_i-\mathbf \mu_x)^T\mathbf w]$$

$$=\mathbf w^T\frac{1}{N}\sum_{i=1}^{N}[(\mathbf x_i-\mathbf \mu_x)(\mathbf x_i-\mathbf \mu_x)^T]\mathbf w$$

$$=\mathbf w^T\mathbf C \mathbf w$$

Aquí, $\mathbf C=\frac{1}{N}\sum_{i=1}^{N}[(\mathbf x_i-\mathbf \mu_x)(\mathbf x_i-\mathbf \mu_x)^T]$ es la matriz de **"covarianza"** de tamaño $d \times d$. 

Recordando en las clases pasadas que la covarianza de dos variables $a$ y $b$ sobre $N$ valores $a_i,b_i, i=1...N$ está dada por $c(a,b) = \frac{1}{N}\sum_{i=1}^{N}[(a_i-\mu_a)(b_i-\mu_b)^T]$ y es una medida de la relación lineal entre dos variables (en este caso dos dimensiones de $\mathbf x$ o dos de nuestras características).

- La covarianza será **alta (positiva)** si el aumento en los valores de una variable por encima de su media se correlaciona con el aumento en los valores de la otra variable por encima de la media de la otra variable. 

- La covarianza será **alta (negativa)** si el aumento en los valores de una variable por encima de su media se correlaciona con la disminución en los valores de la otra variable por debajo de la media de la otra variable.

-  La covarianza será **baja (pequeña positiva o negativa)** si el aumento en los valores de una variable por encima de su media tiene poco efecto o poca correlación con el aumento en los valores de la otra variable por encima de la media de la otra variable.

Normalmente es mejor escalar las variables al mismo rango antes de calcular la covarianza para reducir los efectos de las diferencias en el rango de valores. Esto puede lograrse restando el valor de una variable de su media y dividiendo por su desviación estándar en un proceso llamado normalización media-desviación estándar (o estandarización).

Hasta ahora hemos expresado la varianza de los datos proyectados en términos de $\mathbf w$ usada para la proyección y la matriz de covarianza de los datos que puede calcularse de antemano. Nos interesa encontrar la dirección de máxima varianza ya que la proyección de los datos dados en esa dirección implica la mínima pérdida de información. Ahora, usando el principio de minimización de riesgo estructural, el problema de aprendizaje de encontrar el vector de dirección óptimo $w$ que minimiza la pérdida de información después de la proyección mientras asegura regularización puede escribirse como:

$$min_\mathbf w \alpha R(\mathbf w)+E(\mathbf w)$$

Donde $E$ es el término de error y $R$ controla la (inversa de) regularización. Como el error está inversamente relacionado con la varianza de $z$, podemos escribir este problema de aprendizaje como sigue:

$$min_\mathbf w \alpha ||\mathbf w||^2-var(z) \text{ (para } \alpha \geq 0)$$

o

$$min_\mathbf w \alpha \mathbf w^T\mathbf w -\mathbf w^TC\mathbf w$$

Aquí, queremos minimizar el valor de la norma de $\mathbf w$ mientras maximizamos la varianza.

Tomando la derivada respecto a $\mathbf w^T$ y sustituyendo a cero, obtenemos:

$$\mathbf C\mathbf w=\alpha \mathbf w$$

Nota que hay una solución trivial a esto $\mathbf w = \mathbf 0$. Sin embargo, al restringir $||\mathbf w||=1$, podemos encontrar una solución no trivial. Para este propósito, la ecuación anterior es un problema de valores propios con vector propio $\mathbf w$ y valor propio $\alpha$. 

- Un vector propio $\mathbf w$ es una propiedad de una matriz $\mathbf C$ tal que el vector resultante de la multiplicación matricial $\mathbf C\mathbf w$ está en la misma dirección que $\mathbf w$ solo con un escalamiento por un factor constante correspondiente $\alpha$ llamado valor propio. El número de valores y vectores propios es igual al número de dimensiones de la matriz. Estos vectores de una matriz de covarianza se llaman componentes principales ya que corresponden a direcciones de varianza máxima. De ahí el nombre "Análisis de Componentes Principales".

Así, la dirección de máxima varianza $\mathbf w$ corresponde al vector propio de la matriz de covarianza $\mathbf C$. Así, si encontramos los vectores propios de la matriz de covarianza, ¡podemos obtener lo que queremos! 

### Valores y Vectores Propios

La multiplicación de una matriz por un vector puede interpretarse como una transformación del punto representado en el vector: $$\mathbf y =\mathbf A\mathbf x$$. 

Aquí, el punto $\mathbf x$ ha sido transformado a $\mathbf y$ mediante su multiplicación con la matriz $\mathbf A$. 

La transformación puede ser un escalamiento, proyección, etc., lo que puede cambiar la dirección del punto $\mathbf x$, es decir, los vectores de dirección de $\mathbf x$ y $\mathbf y$ pueden ser diferentes. Sin embargo, existen algunos puntos "característicos" $\mathbf v$ asociados con la matriz, que solo sufren un cambio de escala, es decir, $\mathbf w =\mathbf A\mathbf v = \lambda \mathbf v$. 

Como $\mathbf w = \lambda \mathbf v$,  $\mathbf w$ y $\mathbf v$ son paralelos o ortogonales entre sí, con solo un escalamiento por un factor multiplicativo $\lambda$ resultante de la transformación. Tales puntos o vectores se llaman vectores propios ("eigen" significa "característico" en alemán), mientras que los escalares asociados se llaman valores propios.


### Descomposición en valores propios (Eigen Decomposition)

Dado que:

$$\mathbf w_i = \mathbf T \mathbf v_i = \lambda_i \mathbf v_i$$

Por lo tanto, si $\mathbf V = [\mathbf v_1 | \mathbf v_2, ... | \mathbf v_k]$ y $\mathbf\Lambda$ es una matriz diagonal de valores propios, entonces:

$$\mathbf T \mathbf V= \mathbf V \mathbf\Lambda$$

Así,

$$\mathbf T =\mathbf V \mathbf \Lambda \mathbf V^{-1}$$

Por lo tanto, la matriz original puede escribirse en términos de sus valores y vectores propios, lo que se llama su descomposición en valores propios.

## Resumen de la transformación de PCA

Dado un conjunto de datos $X$ de $n$ observaciones y $p$ variables:

1. **Estandariza los datos:**

$$
Z_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}
$$
donde $\mu_j$ es la media y $\sigma_j$ la desviación estándar de la variable $j$.

2. **Calcula la matriz de covarianza:**

$$
C = \frac{1}{n-1} Z^T Z
$$

3. **Obtén los autovalores ($\lambda$) y autovectores ($v$) de $C$:**

$$
C v = \lambda v
$$

4. **Ordena los autovectores según los autovalores de mayor a menor.**

5. **Proyecta los datos originales en los autovectores seleccionados:**

$$
Y = Z V_k = \mathbf V^T \mathbf Z
$$
donde $V_k$ son los $k$ autovectores principales (componentes principales).

la matriz de componentes principales y su transpuesta son inversas entre sí, es decir, $\mathbf {v}^{-1}=\mathbf {v}^{T}$ o $\mathbf {v}^{T}\mathbf v=\mathbf I$.

Para calcular la proyección inversa. las proyecciones pueden escribirse como $\mathbf Y = \mathbf V^T \mathbf Z$, por lo tanto, la proyección inversa (reproyección) puede escribirse como $\mathbf Z^r = \mathbf {V}^{-1} \mathbf Y$. Ya sabemos que $\mathbf {V}^{-1} = \mathbf {V}^{T}$. Podemos usar esto para calcular la transformación inversa usando $\mathbf Z^r = \mathbf {V}^{-1} \mathbf Y = \mathbf {V}^{T} \mathbf Y $. En forma matricial tenemos $\mathbf Z^r = \mathbf Y \mathbf V^T$.

### PCA Manual con el dataset Iris

Utilizaremos el famoso dataset Iris para ilustrar el proceso manual de PCA paso a paso.

In [ ]:
# Cargar datos 

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)

# 1. Estandarización
X_std = (X - X.mean()) / X.std()
print('Primeras filas de los datos estandarizados:')
print(X_std.head())

In [ ]:
# 2. Matriz de covarianza
cov_matrix = np.cov(X_std.T)
print('Matriz de covarianza:')
print(cov_matrix)

In [ ]:
# 3. Autovalores y eigen valores
eig_vals, eig_vecs = np.linalg.eig(cov_matrix)
print('eigen valores:', eig_vals)
print('eigen vectores (columnas):')
print(eig_vecs)

In [ ]:
# 4. Ordenar autovectores según autovalores
idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:, idx]

# 5. Proyección a los dos primeros componentes principales
X_pca_manual = np.dot(X_std, eig_vecs[:, :2])

# Visualización
plt.figure(figsize=(8,6))
plt.scatter(X_pca_manual[:,0], X_pca_manual[:,1], c=iris.target, cmap='viridis', edgecolor='k')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA manual - Iris')
plt.colorbar(label='Clase')
plt.show()

### PCA usando librerías de Python (scikit-learn)

La librería `scikit-learn` facilita el cálculo de PCA con la clase `PCA`.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

# Visualización
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=iris.target, cmap='viridis', edgecolor='k')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA con scikit-learn - Iris')
plt.colorbar(label='Clase')
plt.show()

### PCA para visualización: Dígitos escritos a mano

La utilidad de la reducción de dimensionalidad puede no ser completamente evidente en solo dos dimensiones, pero se vuelve mucho más clara al observar datos de alta dimensión.
Para ver esto, echemos un vistazo rápido a la aplicación de PCA a los datos de dígitos

In [ ]:
# Load the data
digits_data = datasets.load_digits()
n = len(digits_data.images)


In [ ]:
n

In [ ]:
# Groundtruth label of the number appearing in each image
labels = digits_data.target
labels


Los datos aquí consisten en imágenes de 8×8 píxeles, lo que significa que son de 64 dimensiones.

In [ ]:
# Each image is represented as an 8-by-8 array. 
# Flatten this array as input to PCA.
image_data = digits_data.images.reshape((n, -1))
image_data.shape

In [ ]:
digits_data.images.shape

In [ ]:
digits_data.images[0,:,:]

In [ ]:
plt.imshow(digits_data.images[0,:,:], cmap='grey')

Para obtener una intuición sobre las relaciones entre estos puntos, podemos usar PCA para proyectarlos a un número más manejable de dimensiones, digamos dos:

In [ ]:
pca = PCA(2)  # proyectar de 64 a 2 dimensiones
projected = pca.fit_transform(image_data)
print(image_data.shape)
print(projected.shape)

In [ ]:
#Ahora podemos graficar los dos primeros componentes principales de cada punto para aprender sobre los datos:
plt.figure(figsize=(8,6))
plt.scatter(projected[:, 0], projected[:, 1],
            c=digits_data.target, edgecolor='none', alpha=0.5,
            cmap=plt.get_cmap('winter', 10))
plt.xlabel('componente 1')
plt.ylabel('componente 2')
plt.colorbar();

los datos completos son una nube de puntos de 64 dimensiones, y estos puntos son la proyección de cada punto de datos a lo largo de las direcciones con mayor varianza.
Esencialmente, hemos encontrado el estiramiento y rotación óptimos en el espacio de 64 dimensiones que nos permite ver la disposición de los dígitos en dos dimensiones.

### ¿Qué significan los componentes?

Podemos preguntarnos qué *significan* las dimensiones reducidas.

Este significado puede entenderse en términos de combinaciones de vectores base.

Por ejemplo, cada imagen en el conjunto de entrenamiento se define por una colección de 64 valores de píxeles, que llamaremos el vector $x$:

$$
x = [x_1, x_2, x_3 \cdots x_{64}]
$$

Una forma de pensar en esto es en términos de una base de píxeles.
Es decir, para construir la imagen, multiplicamos cada elemento del vector por el píxel que describe, y luego sumamos los resultados para construir la imagen:

$$
{\rm imagen}(x) = x_1 \cdot{\rm (píxel~1)} + x_2 \cdot{\rm (píxel~2)} + x_3 \cdot{\rm (píxel~3)} \cdots x_{64} \cdot{\rm (píxel~64)}
$$

Una forma en que podríamos imaginar reducir la dimensión de estos datos es eliminando todos menos unos pocos de estos vectores base.
Por ejemplo, si usamos solo los primeros ocho píxeles, obtenemos una proyección de los datos de ocho dimensiones, pero no es muy representativa de toda la imagen: ¡hemos descartado casi el 90% de los píxeles!

![](Figures/05.09-digits-pixel-components.png)

La fila superior muestra los píxeles individuales, y la fila inferior muestra la contribución acumulada de estos píxeles a la construcción de la imagen.
Usando solo ocho de los componentes base de píxeles, solo podemos construir una pequeña parte de la imagen de 64 píxeles.
Si continuáramos esta secuencia y usáramos los 64 píxeles, recuperaríamos la imagen original.

Pero la representación por píxeles no es la única elección de base. También podemos usar otras funciones base, que contienen alguna contribución predefinida de cada píxel, y escribir algo como

$$
imagen(x) = {\rm media} + x_1 \cdot{\rm (base~1)} + x_2 \cdot{\rm (base~2)} + x_3 \cdot{\rm (base~3)} \cdots
$$

PCA puede considerarse como un proceso de elegir funciones base óptimas, de modo que sumar solo las primeras de ellas sea suficiente para reconstruir adecuadamente la mayoría de los elementos en el conjunto de datos.
Los componentes principales, que actúan como la representación de baja dimensión de nuestros datos, son simplemente los coeficientes que multiplican cada uno de los elementos en esta serie.
Esta figura muestra una representación similar de la reconstrucción de este dígito usando la media más las primeras ocho funciones base de PCA:

![](Figures/05.09-digits-pca-components.png)


A diferencia de la base de píxeles, la base de PCA nos permite recuperar las características más importantes de la imagen de entrada con solo la media más ocho componentes.
La cantidad de cada píxel en cada componente es el análogo de la orientación del vector en nuestro ejemplo bidimensional.
En este sentido, PCA proporciona una representación de baja dimensión de los datos.

### ¿Cómo elegir el número de componentes?

Una parte vital de usar PCA en la práctica es la capacidad de estimar cuántos componentes se necesitan para describir los datos.
Esto puede determinarse observando la **varianza explicada acumulada**como función del número de componentes:

In [ ]:
pca = PCA().fit(image_data)

plt.figure(figsize=(8,6))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('número de componentes')
plt.ylabel('varianza explicada acumulada');

Esta curva cuantifica cuánta de la varianza total, de 64 dimensiones, está contenida dentro de los primeros $N$ componentes.
Por ejemplo, vemos que con los dígitos los primeros 10 componentes contienen aproximadamente el 75% de la varianza, mientras que necesitas alrededor de 50 componentes para describir cerca del 100% de la varianza.

Aquí vemos que nuestra proyección bidimensional pierde mucha información (según la varianza explicada) y que necesitaríamos alrededor de 20 componentes para retener el 90% de la varianza. Observar este gráfico para un conjunto de datos de alta dimensión puede ayudar a entender el nivel de redundancia presente en múltiples observaciones.

In [ ]:
#Entrenamos el algoritmo con al menos un 80% total de varianza 
pca_transformer = PCA(n_components=0.8)
pca_images = pca_transformer.fit_transform(image_data)
pca_transformer.explained_variance_ratio_


In [ ]:
pca_transformer.explained_variance_ratio_[:3].sum()


In [ ]:
# Visualize the results
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(6,8))
ax = fig.add_subplot(111, projection='3d')
for i in range(100):
    ax.scatter(pca_images[i,0], pca_images[i,1], pca_images[i,2],
    marker=f'${labels[i]}$', s=30)
 
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')


### PCA como filtrado de ruido

PCA también puede usarse como un enfoque de filtrado para datos ruidosos.

La idea es esta: 
- cualquier componente con varianza mucho mayor que el efecto del ruido debería ser relativamente poco afectado por el ruido.

Así que si se reconstruye los datos usando solo el subconjunto más grande de componentes principales, se debería mantener preferentemente la señal y el ruido sería eliminado.


In [ ]:
#Datos originales sin ruido:
def plot_digits(data):
    fig, axes = plt.subplots(4, 10, figsize=(10, 4),
                             subplot_kw={'xticks':[], 'yticks':[]},
                             gridspec_kw=dict(hspace=0.1, wspace=0.1))
    for i, ax in enumerate(axes.flat):
        ax.imshow(data[i].reshape(8, 8),
                  cmap='binary', interpolation='nearest',
                  clim=(0, 16))

plot_digits(image_data)

In [ ]:
# Ahora agreguemos algo de ruido aleatorio
np.random.seed(42)
image_data_noisy = np.random.normal(image_data, 4)
plot_digits(image_data_noisy)

In [ ]:
# Entrenemos un PCA en los datos ruidosos, solicitando que la proyección preserve el 50% de la varianza:
pca = PCA(0.50).fit(image_data_noisy)
pca.n_components_

In [ ]:
# Con 12 componentes alcanzamos el 50% de varianza acumulada

In [ ]:
components = pca.transform(image_data_noisy)
filtered = pca.inverse_transform(components)
plot_digits(filtered)

Esta propiedad de preservar la señal/filtrar el ruido hace que PCA sea una rutina muy útil para la selección de características, por ejemplo, en lugar de entrenar un clasificador en datos de muy alta dimensión, podrías entrenar el clasificador en la representación de menor dimensión, lo que automáticamente servirá para filtrar el ruido aleatorio en las entradas.

## Resumen de PCA

- **Reducción de dimensionalidad:** Permite representar los datos en menos dimensiones conservando la mayor parte de la varianza.
- **Eliminación de redundancia:** Los componentes principales son ortogonales (no correlacionados).
- **Visualización:** Facilita la visualización de datos multidimensionales.

#### Riesgos
- Los componentes principales pueden ser difíciles de interpretar.
- PCA es una transformación lineal: no captura relaciones no lineales.
- PCA reduce la dimensionalidad del espacio de características buscando patrones de correlación lineal entre las características. Dado que implica la descomposición en valores singulares (SVD), el cálculo de PCA es costoso para más de unos pocos miles de características. Sin embargo, para un pequeño número de características con valores reales, definitivamente vale la pena intentarlo.
